# Lab 5: LLM-as-Judge — Automated Evaluation

**Series**: Agentic AI Science Playbook — Foundation Layer
**Goal**: Build a scalable automated evaluation harness using a separate LLM as judge.
**Time**: ~45 min.

## Why LLM-as-Judge?

```
 Agent Output --> Judge LLM (rubric) --> Score + Rationale --> Decision
```

Scales to thousands of examples. Use for A/B testing prompts, models, and tool configurations.

**Caveat**: LLM judges have biases. Calibrate against human labels on a small sample.

## Learning Objectives

By the end of this lab, you will be able to:
- Design structured evaluation rubrics using Pydantic schemas
- Implement automated quality assessment with a separate LLM judge
- Run A/B tests comparing different agent configurations
- Understand the strengths and limitations of LLM-based evaluation

## Why This Matters for Scientists

In science, **every measurement needs a metric**. You wouldn't publish experimental results without statistical tests. Similarly, you shouldn't deploy an agent without systematic quality evaluation.

LLM-as-Judge provides a **scalable, automated quality signal** that lets you:
- Compare prompt variants objectively (A/B testing)
- Detect quality regressions when you change models or tools
- Measure domain-specific accuracy (correctness, groundedness, completeness)

> **Real-world application**: NVIDIA uses LLM-as-Judge evaluation harnesses to benchmark NIM models across scientific domains. The AI-Q Blueprint scored #1 on the DeepResearch Bench using exactly this pattern — automated multi-dimensional evaluation of agent outputs.

## Prerequisites & NVIDIA SDK Connection

| Requirement | Details |
|------------|---------|
| Completed | Labs 0-4 |
| Concepts | Pydantic schemas, agent loops |

**NVIDIA Connection**: The **NeMo Agent Toolkit** includes built-in evaluation and profiling capabilities that extend this pattern to production. You can use the same rubric approach to evaluate agents running on different NIM models (e.g., Nemotron 49B vs. 70B) and find the best quality/cost tradeoff for your scientific domain.

### Install Dependencies
We install the OpenAI client and Pydantic for structured evaluation schemas.

In [ ]:
!pip install -q openai pydantic

### Connect to LLM
Same dual-provider setup. We also import `json` and `pydantic` for structured output parsing.

> **Why NIM?** Data privacy, faster inference, open models. See Lab 0 for the full comparison.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI
use_nim = os.environ.get("USE_NIM","").lower() in ("1","true","yes") or "NIM_API_KEY" in os.environ
if use_nim:
    if "NIM_API_KEY" not in os.environ: os.environ["NIM_API_KEY"]=getpass("NVIDIA NIM API key: ")
    client=OpenAI(base_url="https://integrate.api.nvidia.com/v1",api_key=os.environ["NIM_API_KEY"])
    MODEL=os.environ.get("NIM_MODEL","nvidia/llama-3.3-nemotron-super-49b-v1.5")
else:
    if "OPENAI_API_KEY" not in os.environ: os.environ["OPENAI_API_KEY"]=getpass("OpenAI API key: ")
    client=OpenAI()
    MODEL="gpt-4o-mini"
print(f"Using model: {MODEL}")
import json
from pydantic import BaseModel, Field

## Concept: Evaluation as a Pydantic Schema

Why use a Pydantic schema for evaluation? The same reasons you use schemas for tools:

1. **Structured output**: The judge returns typed scores, not free text
2. **Consistency**: Every evaluation uses the same dimensions
3. **Aggregation**: Numerical scores can be averaged, compared, and plotted

### The Four Dimensions

| Dimension | What It Measures | Why Scientists Care |
|-----------|-----------------|---------------------|
| **Correctness** | Are facts accurate? | Wrong facts → wrong conclusions |
| **Completeness** | Are all aspects covered? | Missing info → incomplete analysis |
| **Groundedness** | Are claims evidence-backed? | Ungrounded claims → hallucination |
| **Clarity** | Is output well-structured? | Unclear output → misinterpretation |

> **Customization**: For your domain, you might add dimensions like `statistical_rigor`, `citation_quality`, or `protocol_completeness`. The Pydantic schema makes it easy to extend.

### Define the Evaluation Rubric
We use a Pydantic schema to define exactly how the judge should score agent outputs. Each dimension is scored 1-5, plus a text rationale explaining the scores.

## Evaluation Rubric

In [ ]:
class EvaluationResult(BaseModel):
    correctness:  int = Field(..., ge=1, le=5, description="1=hallucinated, 5=fully accurate")
    completeness: int = Field(..., ge=1, le=5, description="1=missing key info, 5=fully covers question")
    groundedness: int = Field(..., ge=1, le=5, description="1=unsupported, 5=evidence-backed")
    clarity:      int = Field(..., ge=1, le=5, description="1=confusing, 5=clear and structured")
    rationale: str = Field(..., description="2-4 sentence explanation of the scores.")

    @property
    def overall(self) -> float:
        return (self.correctness + self.completeness + self.groundedness + self.clarity) / 4

print("Rubric dimensions: correctness, completeness, groundedness, clarity (1-5 each)")

## Judge Function

### Build the Judge Function
The judge is itself an LLM call. It takes a question and agent response, evaluates against our rubric, and returns structured scores. The key: `response_format={'type': 'json_object'}` ensures we get parseable JSON back.

In [ ]:
JUDGE_SYSTEM = (
    "You are an expert scientific evaluator. Assess responses on: "
    "correctness, completeness, groundedness, clarity (each 1-5). "
    "Return ONLY valid JSON matching the EvaluationResult schema. No other text."
)

def judge(question: str, agent_response: str, context: str = "") -> EvaluationResult:
    ctx = f"\nContext:\n{context}" if context else ""
    prompt = f"Question: {question}\nAgent response: {agent_response}{ctx}\n\nReturn JSON evaluation."
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0, max_tokens=400,
        response_format={"type": "json_object"},
        messages=[{"role": "system", "content": JUDGE_SYSTEM},
                   {"role": "user", "content": prompt}])
    data = json.loads(r.choices[0].message.content or "{}")
    return EvaluationResult(**data)

## Evaluate Three Response Types

### Evaluate Three Response Types
We test the judge on three responses to the same question about CRISPR:
- **GOOD**: Accurate, complete, cited
- **POOR**: Vague, no specifics
- **HALLUCINATED**: Confident but factually wrong

Watch how the judge assigns different scores to each.

In [ ]:
QUESTION = "What is the mechanism of action of CRISPR-Cas9 in genome editing?"

RESPONSES = {
    "GOOD": (
        "CRISPR-Cas9 uses a guide RNA (sgRNA) to direct the Cas9 endonuclease to a specific genomic locus. "
        "Cas9 creates a double-strand break repaired via NHEJ (disruption) or HDR (precise insertion). "
        "Originally discovered as bacterial adaptive immunity (Jinek et al., Science 2012)."
    ),
    "POOR": "CRISPR works by cutting DNA. Scientists use it to change genes. It is very useful.",
    "HALLUCINATED": (
        "CRISPR-Cas9 uses quantum entanglement between guide RNA and target DNA. "
        "Cas12 directly rewrites genetic code without creating any breaks. "
        "Invented in 2019, FDA-approved for all therapeutic uses."
    ),
}

for label, response in RESPONSES.items():
    r = judge(QUESTION, response)
    print(f"\n--- {label} ---")
    print(f"  Correctness  : {r.correctness}/5")
    print(f"  Completeness : {r.completeness}/5")
    print(f"  Groundedness : {r.groundedness}/5")
    print(f"  Clarity      : {r.clarity}/5")
    print(f"  OVERALL      : {r.overall:.2f}/5")
    print(f"  Rationale    : {r.rationale[:150]}")

<details>
<summary>Expected output (click to expand)</summary>

```
--- GOOD ---
  Correctness  : 5/5
  Completeness : 4/5
  Groundedness : 5/5
  Clarity      : 5/5
  OVERALL      : 4.75/5
  Rationale    : The response accurately describes the CRISPR-Cas9 mechanism...

--- POOR ---
  Correctness  : 3/5
  Completeness : 1/5
  Groundedness : 1/5
  Clarity      : 3/5
  OVERALL      : 2.00/5
  Rationale    : While technically not wrong, the response is extremely vague...

--- HALLUCINATED ---
  Correctness  : 1/5
  Completeness : 2/5
  Groundedness : 1/5
  Clarity      : 4/5
  OVERALL      : 2.00/5
  Rationale    : Contains multiple factual errors: quantum entanglement is not involved...
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

## Concept: A/B Testing Agent Configurations

A/B testing for agents is like A/B testing for experiments — you change ONE variable and measure the effect:

| Variable to Test | Agent A | Agent B |
|-----------------|---------|---------|
| System prompt | Generic ("helpful assistant") | Domain-specific ("expert biologist") |
| Model | gpt-4o-mini | nvidia/nemotron-super-49b |
| Temperature | 0.0 | 0.3 |
| Tools | Basic tool set | Expanded tool set |

The LLM-as-Judge evaluates both outputs on the same rubric, giving you an **objective comparison** that doesn't depend on human reviewers (who are expensive, slow, and inconsistent).

> **Caveat**: LLM judges have biases — they tend to prefer longer, more confident-sounding responses. Always calibrate against human labels on a small sample before trusting automated scores.

## A/B Test Two Prompt Variants

### Define Two Agent Variants for A/B Testing
**Agent A** uses a generic system prompt ('helpful assistant'). **Agent B** uses a domain-specific prompt ('expert molecular biologist, cite papers'). We'll see which produces better-quality answers.

In [ ]:
def agent_A(q):
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0, max_tokens=200,
        messages=[{"role": "system", "content": "You are a helpful assistant."},
                   {"role": "user", "content": q}])
    return (r.choices[0].message.content or "").strip()

def agent_B(q):
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0, max_tokens=200,
        messages=[{"role": "system", "content": (
            "You are an expert molecular biologist. Provide accurate, well-structured answers. "
            "Cite key papers or authors. Distinguish established facts from emerging findings.")},
                   {"role": "user", "content": q}])
    return (r.choices[0].message.content or "").strip()

### Run the A/B Test
We test both agents on 3 biology questions, score each with our judge, and compare mean scores. This tells us objectively whether the domain-specific prompt improves quality.

In [ ]:
questions = [
    "Explain the role of telomerase in cellular aging.",
    "What are the main mechanisms of antibiotic resistance?",
    "How does RNA sequencing work?",
]
scores_A, scores_B = [], []
for q in questions:
    eA = judge(q, agent_A(q))
    eB = judge(q, agent_B(q))
    scores_A.append(eA.overall)
    scores_B.append(eB.overall)
    winner = "B" if eB.overall > eA.overall else "A" if eA.overall > eB.overall else "tie"
    print(f"Q: {q[:60]}")
    print(f"  A={eA.overall:.2f}  B={eB.overall:.2f}  Winner={winner}\n")

print(f"Mean A={sum(scores_A)/len(scores_A):.2f}  Mean B={sum(scores_B)/len(scores_B):.2f}")

<details>
<summary>Expected output (click to expand)</summary>

```
Q: Explain the role of telomerase in cellular aging.
  A=3.75  B=4.50  Winner=B

Q: What are the main mechanisms of antibiotic resistance?
  A=3.50  B=4.25  Winner=B

Q: How does RNA sequencing work?
  A=3.75  B=4.50  Winner=B

Mean A=3.67  Mean B=4.42
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

## Reflection Questions

1. **What evaluation dimensions** would be most important for agents in your scientific domain? Would you add `statistical_rigor`, `safety`, or `novelty`?
2. **LLM judges have biases.** How would you detect and correct for these biases in your evaluation pipeline?
3. **You A/B tested two prompt variants.** What other agent configurations would be valuable to A/B test for your use case?

## Congratulations!

You've completed all 6 Foundation Labs. You now have production-grade patterns for:

| Lab | Pattern | Skill |
|-----|---------|-------|
| 0 | Minimal agent loop | Build agents from scratch |
| 1 | Prompt-driven decisions | Engineer reliable prompts |
| 2 | Type-safe tool contracts | Define robust tool interfaces |
| 3 | Three-layer memory | Build persistent agents |
| 4 | Graph-based orchestration | Handle complex workflows |
| 5 | Automated evaluation | Measure and improve quality |

**Next step**: Choose a domain track — EOP, Healthcare, or Bioinformatics — and apply these patterns to real science!

## Summary

| Capability | How it works |
|------------|-------------|
| Rubric | Pydantic schema defines 4 dimensions (1-5) |
| Judge | LLM fills schema for any (question, response) pair |
| A/B test | Compare mean scores across prompt variants |
| Calibration | Spot-check against human labels |

**Next**: Choose a domain track — EOP, Healthcare, or Bioinformatics.

---
*Agentic AI Science Playbook — Foundation Layer, Lab 5.*